In [1]:
#export MODEL="Your model here
export MODEL="unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL"

In [2]:
llama-fit-params -hf $MODEL

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping

common_params_fit_impl: getting device memory data for initial parameters:[K
common_memory_breakdown_print: | memory breakdown [MiB] | total   free    self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 5232 + (7135 =  4241 +    2088 +     806) + 17592186037785 |
common_memory_breakdown_print: |   - Host               |                 3825 =  3536 +       0 +     289                   |
common_params_fit_impl: projected to use 7135 MiB of device memory vs. 5232 MiB of free device memory
common_params_fit_impl: cannot meet free memory target of 1024 MiB, need

In [ ]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256 512"
FITT="256 512"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            # Get fitted parameters  
            #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  \
            
            # need to add fitt for nvidia
            OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft 2>&1)
            
            #echo "Raw output: $OUTPUT"  # Debug line  
            
            # Extract context and GPU layers more robustly  
            CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
            NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
            
            if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                echo "ub: $ub, b: $b, fitt: $ft, ctx: $CTX, ngl: $NGL"  
            else  
                echo "No valid parameters found"  
            fi 
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL...
ub: 64, b: 1024, fitt: 256, ctx: 13568, ngl: -1
ub: 64, b: 1024, fitt: 512, ctx: 4096, ngl: 42
ub: 64, b: 2048, fitt: 256, ctx: 16640, ngl: -1
ub: 64, b: 2048, fitt: 512, ctx: 4096, ngl: 42
ub: 128, b: 1024, fitt: 256, ctx: 12288, ngl: -1
ub: 128, b: 1024, fitt: 512, ctx: 4096, ngl: 42
ub: 128, b: 2048, fitt: 256, ctx: 12288, ngl: -1
ub: 128, b: 2048, fitt: 512, ctx: 4096, ngl: 42
ub: 256, b: 1024, fitt: 256, ctx: 4096, ngl: 42
ub: 256, b: 1024, fitt: 512, ctx: 4096, ngl: 40
ub: 256, b: 2048, fitt: 256, ctx: 4096, ngl: 42
ub: 256, b: 2048, fitt: 512, ctx: 4096, ngl: 40
ub: 512, b: 1024, fitt: 256, ctx: 4096, ngl: 40
ub: 512, b: 1024, fitt: 512, ctx: 4096, ngl: 37


In [3]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -ub 64 -ngl 99 -p 1000 -n 256

Staring llama-bench for unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl | n_ubatch |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | -------: | --------------: | -------------------: |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA       |  99 |       64 |          pp1000 |      1272.46 ± 37.74 |
| gemma4 E4B Q6_K                |   6.93 GiB |     7.52 B | CUDA       |  99 |       64 |           tg256 |         50.27 ± 0.93 |

build: 0adede866 (8925)


In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

In [ ]:
echo "Staring llama-server for ${MODEL}..." 
llama-server -hf $MODEL --threads-http 1 -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off -np 1